#

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
import requests

In [2]:
f = "../data/3960AFa8.csv"
df = pd.read_csv(f, skiprows=3)
energy_columns = [col for col in df.columns if col not in ['Año', 'Unidades']]
fig = px.area(df, x='Año', y=energy_columns, labels={'value': 'Energía primaria [TJ]', 'variable': 'Fuente'})
fig.update_layout(hovermode='x unified')
fig.show()

In [3]:
f = "../data/3960AFa8.csv"
df = pd.read_csv(f, skiprows=3)
energy_columns = [col for col in df.columns if col not in ['Año', 'Unidades']]
fig = px.area(df, x='Año', y=energy_columns, groupnorm='percent', labels={'value': '%', 'variable': 'Fuente'})
fig.update_layout(hovermode='x unified')
fig.show()

In [4]:
f = "../data/primary-energy-consumption-by-region.csv"
df2 = pd.read_csv(f)
region_mapping = {
    'Africa (EI)': 'África',
    'Asia Pacific (EI)': 'Asia Pacífico',
    'Europe (EI)': 'Europa',
    'Middle East (EI)': 'Medio Oriente',
    'North America (EI)': 'Norteamérica',
    'South and Central America (EI)': 'Sudamérica y Centroamérica'
}
df2['Entity'] = df2['Entity'].replace(region_mapping)
fig = px.area(df2, x='Year', y='Primary energy consumption', color='Entity', labels={'Year': 'Año', 'Primary energy consumption': 'Energía primaria [TJ]', 'Entity': 'Región'})
fig.update_layout(hovermode='x unified')
fig.show()

In [5]:
f = "../data/primary-energy-consumption-by-region.csv"
df2 = pd.read_csv(f)
region_mapping = {
    'Africa (EI)': 'África',
    'Asia Pacific (EI)': 'Asia Pacífico',
    'Europe (EI)': 'Europa',
    'Middle East (EI)': 'Medio Oriente',
    'North America (EI)': 'Norteamérica',
    'South and Central America (EI)': 'Sudamérica y Centroamérica'
}
df2['Entity'] = df2['Entity'].replace(region_mapping)
fig = px.area(df2, x='Year', y='Primary energy consumption', color='Entity', groupnorm='percent', labels={'Year': 'Año', 'Primary energy consumption': '%', 'Entity': 'Región'})
fig.update_layout(hovermode='x unified')
fig.show()

In [6]:
f = "../data/iea_renovables_mundo_capacidad_es.csv"
df3 = pd.read_csv(f)
df3_series = df3[df3['producto'] != 'Total'].copy()
df3_main = df3_series[df3_series['escenario'] == 'Caso principal'].copy()
df3_acc = df3_series[df3_series['escenario'] == 'Caso acelerado'].copy()
df3_target = df3[df3['escenario'] == 'Meta'].copy()
productos = df3_main['producto'].drop_duplicates().tolist()
colors = px.colors.qualitative.Plotly
color_map = {p: colors[i % len(colors)] for i, p in enumerate(productos)}
main_total = df3_main.groupby('anio', as_index=False)['valor'].sum()
acc_total = df3_acc.groupby('anio', as_index=False)['valor'].sum()
fig = go.Figure()
for p in productos:
    main_p = df3_main[df3_main['producto'] == p].sort_values('anio')
    acc_p = df3_acc[df3_acc['producto'] == p].sort_values('anio')
    fig.add_trace(go.Bar(x=main_p['anio'], y=main_p['valor'], name=p, legendgroup=p, offsetgroup='Caso principal', marker={'color': color_map[p]}))
    fig.add_trace(go.Bar(x=acc_p['anio'], y=acc_p['valor'], name=p, legendgroup=p, offsetgroup='Caso acelerado', showlegend=False, marker={'color': color_map[p], 'pattern': {'shape': '/'}}))
fig.add_hline(y=11500, line_dash='dash', line_color='black', annotation_text='Meta COP28: triplicar renovables para 2030', annotation_position='top left')
fig.add_trace(go.Scatter(x=df3_target['anio'], y=df3_target['valor'], mode='markers+text', name='Ambición renovable actual 2030', text=['Ambición renovable actual 2030'], textposition='top center', marker={'size': 10, 'symbol': 'diamond', 'color': 'red'}))
main_2025 = main_total[main_total['anio'] == 2025]['valor'].iloc[0]
acc_2025 = acc_total[acc_total['anio'] == 2025]['valor'].iloc[0]
fig.add_annotation(x=2025, y=main_2025, text='Caso principal', showarrow=False, yshift=16, xshift=-34)
fig.add_annotation(x=2025, y=acc_2025, text='Caso acelerado', showarrow=False, yshift=16, xshift=34)
fig.update_layout(barmode='stack', hovermode='x unified', xaxis_title='Año', yaxis_title='Capacidad instalada [GW]', legend_title='Tecnología')
fig.show()

In [7]:
df = pd.read_csv('https://ourworldindata.org/grapher/per-capita-oil.csv?v=1&csvType=full&useColumnShortNames=true', storage_options={'User-Agent': 'Our World In Data data fetch/1.0'})
regiones = ['Africa', 'Asia', 'Europe', 'North America', 'Oceania', 'South America', 'World']
tr = {'Africa': 'África', 'Asia': 'Asia', 'Europe': 'Europa', 'North America': 'Norteamérica', 'Oceania': 'Oceanía', 'South America': 'Sudamérica', 'World': 'Mundo'}
df_oil = df[df['entity'].isin(regiones)].copy()
df_oil['entity'] = df_oil['entity'].replace(tr)
fig = px.line(df_oil, x='year', y='oil_per_capita__kwh', color='entity', labels={'year': 'Año', 'oil_per_capita__kwh': 'Consumo de petróleo [kWh per cápita]', 'entity': 'Región'})
fig.update_layout(hovermode='x unified')
fig.show()

In [8]:
df = pd.read_csv('https://ourworldindata.org/grapher/electricity-prod-source-stacked.csv?v=1&csvType=full&useColumnShortNames=true', storage_options={'User-Agent': 'Our World In Data data fetch/1.0'})
df_elec = df[df['entity'] == 'World'].copy()
col_map = {
    'other_renewables_excluding_bioenergy_generation__twh_chart_electricity_prod_source_stacked': 'Otras renovables (sin bioenergía)',
    'bioenergy_generation__twh_chart_electricity_prod_source_stacked': 'Bioenergía',
    'solar_generation__twh_chart_electricity_prod_source_stacked': 'Solar',
    'wind_generation__twh_chart_electricity_prod_source_stacked': 'Eólica',
    'hydro_generation__twh_chart_electricity_prod_source_stacked': 'Hidroeléctrica',
    'nuclear_generation__twh_chart_electricity_prod_source_stacked': 'Nuclear',
    'oil_generation__twh_chart_electricity_prod_source_stacked': 'Petróleo',
    'gas_generation__twh_chart_electricity_prod_source_stacked': 'Gas',
    'coal_generation__twh_chart_electricity_prod_source_stacked': 'Carbón'
}
source_cols = [c for c in df_elec.columns if c in col_map]
df_elec = df_elec.rename(columns=col_map)
source_cols_es = [col_map[c] for c in source_cols]
fig = px.area(df_elec, x='year', y=source_cols_es, labels={'year': 'Año', 'value': 'Generación eléctrica [TWh]', 'variable': 'Fuente'})
fig.update_layout(hovermode='x unified')
fig.show()

In [9]:
df = pd.read_csv('https://ourworldindata.org/grapher/electricity-prod-source-stacked.csv?v=1&csvType=full&useColumnShortNames=true', storage_options={'User-Agent': 'Our World In Data data fetch/1.0'})
df_elec = df[df['entity'] == 'World'].copy()
col_map = {
    'other_renewables_excluding_bioenergy_generation__twh_chart_electricity_prod_source_stacked': 'Otras renovables (sin bioenergía)',
    'bioenergy_generation__twh_chart_electricity_prod_source_stacked': 'Bioenergía',
    'solar_generation__twh_chart_electricity_prod_source_stacked': 'Solar',
    'wind_generation__twh_chart_electricity_prod_source_stacked': 'Eólica',
    'hydro_generation__twh_chart_electricity_prod_source_stacked': 'Hidroeléctrica',
    'nuclear_generation__twh_chart_electricity_prod_source_stacked': 'Nuclear',
    'oil_generation__twh_chart_electricity_prod_source_stacked': 'Petróleo',
    'gas_generation__twh_chart_electricity_prod_source_stacked': 'Gas',
    'coal_generation__twh_chart_electricity_prod_source_stacked': 'Carbón'
}
source_cols = [c for c in df_elec.columns if c in col_map]
df_elec = df_elec.rename(columns=col_map)
source_cols_es = [col_map[c] for c in source_cols]
fig = px.area(df_elec, x='year', y=source_cols_es, groupnorm='percent', labels={'year': 'Año', 'value': '%', 'variable': 'Fuente'})
fig.update_layout(hovermode='x unified')
fig.show()

In [10]:
df = pd.read_csv('https://ourworldindata.org/grapher/per-capita-electricity-generation.csv?v=1&csvType=full&useColumnShortNames=true', storage_options={'User-Agent': 'Our World In Data data fetch/1.0'})
_ = requests.get('https://ourworldindata.org/grapher/per-capita-electricity-generation.metadata.json?v=1&csvType=full&useColumnShortNames=true').json()
regiones = ['Africa', 'Asia', 'Europe', 'North America', 'Oceania', 'South America', 'World']
tr = {'Africa': 'África', 'Asia': 'Asia', 'Europe': 'Europa', 'North America': 'Norteamérica', 'Oceania': 'Oceanía', 'South America': 'Sudamérica', 'World': 'Mundo'}
df_reg = df[df['entity'].isin(regiones)].copy()
df_reg = df_reg[df_reg['year'] <= 2023].copy()
df_reg['region'] = df_reg['entity'].replace(tr)
value_col = 'per_capita_electricity_generation__kwh'
if value_col not in df_reg.columns:
    candidate_cols = [c for c in df_reg.columns if c not in ['entity', 'code', 'year']]
    value_col = next((c for c in candidate_cols if 'per_capita' in c), candidate_cols[0])
fig = px.area(df_reg.sort_values(['region', 'year']), x='year', y=value_col, color='region', labels={'year': 'Año', value_col: 'Generación eléctrica per cápita [kWh]', 'region': 'Región'})
fig.update_layout(hovermode='x unified')
fig.show()

In [11]:
df = pd.read_csv('https://ourworldindata.org/grapher/per-capita-electricity-generation.csv?v=1&csvType=full&useColumnShortNames=true', storage_options={'User-Agent': 'Our World In Data data fetch/1.0'})
regiones = ['Africa', 'Asia', 'Europe', 'North America', 'Oceania', 'South America', 'World']
tr = {'Africa': 'África', 'Asia': 'Asia', 'Europe': 'Europa', 'North America': 'Norteamérica', 'Oceania': 'Oceanía', 'South America': 'Sudamérica', 'World': 'Mundo'}
df_reg = df[df['entity'].isin(regiones)].copy()
df_reg = df_reg[df_reg['year'] <= 2023].copy()
df_reg['region'] = df_reg['entity'].replace(tr)
value_col = 'per_capita_electricity_generation__kwh'
if value_col not in df_reg.columns:
    candidate_cols = [c for c in df_reg.columns if c not in ['entity', 'code', 'year']]
    value_col = next((c for c in candidate_cols if 'per_capita' in c), candidate_cols[0])
fig = px.area(df_reg.sort_values(['region', 'year']), x='year', y=value_col, color='region', groupnorm='percent', labels={'year': 'Año', value_col: '%', 'region': 'Región'})
fig.update_layout(hovermode='x unified')
fig.show()

In [12]:
f = '../data/A_A2_r_230822.081459.xlsx'
df_raw = pd.read_excel(f, sheet_name='A2', skiprows=5)
year_cols = [c for c in df_raw.columns if isinstance(c, (int, float))]
df_raw = df_raw[df_raw['Region and fuel'].notna()].copy()
df_raw['Region and fuel'] = df_raw['Region and fuel'].astype(str)
df_raw = df_raw[~df_raw['Region and fuel'].str.startswith('Data source:')].copy()
rows = []
current_region = None
for _, row in df_raw.iterrows():
    label = row['Region and fuel'].strip()
    values = row[year_cols]
    if values.isna().all():
        current_region = label
        continue
    rows.append({'region': current_region, 'fuel': label, **{int(y): row[y] for y in year_cols}})
df_long = pd.DataFrame(rows).melt(id_vars=['region', 'fuel'], var_name='year', value_name='consumo_quad_btu')
df_long['consumo_kwh'] = df_long['consumo_quad_btu'] * 2.9307107e11
fuel_map = {'Liquid fuels': 'Combustibles líquidos', 'Natural gas': 'Gas natural', 'Coal': 'Carbón', 'Nuclear': 'Nuclear', 'Other': 'Otras', 'Total': 'Total'}
df_long['fuel_es'] = df_long['fuel'].replace(fuel_map)
df_fuente = df_long[(df_long['region'] == 'World') & (df_long['fuel'] != 'Total')].copy()
fig = px.area(df_fuente.sort_values(['fuel_es', 'year']), x='year', y='consumo_kwh', color='fuel_es', labels={'year': 'Año', 'consumo_kwh': 'Consumo de energía primaria [kWh]', 'fuel_es': 'Fuente'})
fig.add_vline(x=2026, line_dash='dash', line_color='black')
fig.add_annotation(x=2025.5, y=1.04, xref='x', yref='paper', text='Histórico', showarrow=False, xanchor='right')
fig.add_annotation(x=2026.5, y=1.04, xref='x', yref='paper', text='Proyección', showarrow=False, xanchor='left')
fig.update_layout(hovermode='x unified')
fig.show()

In [13]:
f = '../data/A_A2_r_230822.081459.xlsx'
df_raw = pd.read_excel(f, sheet_name='A2', skiprows=5)
year_cols = [c for c in df_raw.columns if isinstance(c, (int, float))]
df_raw = df_raw[df_raw['Region and fuel'].notna()].copy()
df_raw['Region and fuel'] = df_raw['Region and fuel'].astype(str)
df_raw = df_raw[~df_raw['Region and fuel'].str.startswith('Data source:')].copy()
rows = []
current_region = None
for _, row in df_raw.iterrows():
    label = row['Region and fuel'].strip()
    values = row[year_cols]
    if values.isna().all():
        current_region = label
        continue
    rows.append({'region': current_region, 'fuel': label, **{int(y): row[y] for y in year_cols}})
df_long = pd.DataFrame(rows).melt(id_vars=['region', 'fuel'], var_name='year', value_name='consumo_quad_btu')
df_long['consumo_kwh'] = df_long['consumo_quad_btu'] * 2.9307107e11
fuel_map = {'Liquid fuels': 'Combustibles líquidos', 'Natural gas': 'Gas natural', 'Coal': 'Carbón', 'Nuclear': 'Nuclear', 'Other': 'Otras', 'Total': 'Total'}
df_long['fuel_es'] = df_long['fuel'].replace(fuel_map)
df_fuente = df_long[(df_long['region'] == 'World') & (df_long['fuel'] != 'Total')].copy()
fig = px.area(df_fuente.sort_values(['fuel_es', 'year']), x='year', y='consumo_kwh', color='fuel_es', groupnorm='percent', labels={'year': 'Año', 'consumo_kwh': '%', 'fuel_es': 'Fuente'})
fig.add_vline(x=2026, line_dash='dash', line_color='black')
fig.add_annotation(x=2025.5, y=1.04, xref='x', yref='paper', text='Histórico', showarrow=False, xanchor='right')
fig.add_annotation(x=2026.5, y=1.04, xref='x', yref='paper', text='Proyección', showarrow=False, xanchor='left')
fig.update_layout(hovermode='x unified')
fig.show()

In [14]:
f = '../data/A_A2_r_230822.081459.xlsx'
df_raw = pd.read_excel(f, sheet_name='A2', skiprows=5)
year_cols = [c for c in df_raw.columns if isinstance(c, (int, float))]
df_raw = df_raw[df_raw['Region and fuel'].notna()].copy()
df_raw['Region and fuel'] = df_raw['Region and fuel'].astype(str)
df_raw = df_raw[~df_raw['Region and fuel'].str.startswith('Data source:')].copy()
rows = []
current_region = None
for _, row in df_raw.iterrows():
    label = row['Region and fuel'].strip()
    values = row[year_cols]
    if values.isna().all():
        current_region = label
        continue
    rows.append({'region': current_region, 'fuel': label, **{int(y): row[y] for y in year_cols}})
df_long = pd.DataFrame(rows).melt(id_vars=['region', 'fuel'], var_name='year', value_name='consumo_quad_btu')
df_long['consumo_kwh'] = df_long['consumo_quad_btu'] * 2.9307107e11
region_map = {'Americas': 'Américas', 'Europe and Eurasia': 'Europa y Eurasia', 'Asia Pacific': 'Asia Pacífico', 'Africa and Middle East': 'África y Medio Oriente', 'World': 'Mundo'}
df_long['region_es'] = df_long['region'].replace(region_map)
df_region = df_long[(df_long['fuel'] == 'Total') & (df_long['region'] != 'World')].copy()
fig = px.area(df_region.sort_values(['region_es', 'year']), x='year', y='consumo_kwh', color='region_es', labels={'year': 'Año', 'consumo_kwh': 'Consumo de energía primaria [kWh]', 'region_es': 'Región'})
fig.add_vline(x=2026, line_dash='dash', line_color='black')
fig.add_annotation(x=2025.5, y=1.04, xref='x', yref='paper', text='Histórico', showarrow=False, xanchor='right')
fig.add_annotation(x=2026.5, y=1.04, xref='x', yref='paper', text='Proyección', showarrow=False, xanchor='left')
fig.update_layout(hovermode='x unified')
fig.show()

In [15]:
f = '../data/A_A2_r_230822.081459.xlsx'
df_raw = pd.read_excel(f, sheet_name='A2', skiprows=5)
year_cols = [c for c in df_raw.columns if isinstance(c, (int, float))]
df_raw = df_raw[df_raw['Region and fuel'].notna()].copy()
df_raw['Region and fuel'] = df_raw['Region and fuel'].astype(str)
df_raw = df_raw[~df_raw['Region and fuel'].str.startswith('Data source:')].copy()
rows = []
current_region = None
for _, row in df_raw.iterrows():
    label = row['Region and fuel'].strip()
    values = row[year_cols]
    if values.isna().all():
        current_region = label
        continue
    rows.append({'region': current_region, 'fuel': label, **{int(y): row[y] for y in year_cols}})
df_long = pd.DataFrame(rows).melt(id_vars=['region', 'fuel'], var_name='year', value_name='consumo_quad_btu')
df_long['consumo_kwh'] = df_long['consumo_quad_btu'] * 2.9307107e11
region_map = {'Americas': 'Américas', 'Europe and Eurasia': 'Europa y Eurasia', 'Asia Pacific': 'Asia Pacífico', 'Africa and Middle East': 'África y Medio Oriente', 'World': 'Mundo'}
df_long['region_es'] = df_long['region'].replace(region_map)
df_region = df_long[(df_long['fuel'] == 'Total') & (df_long['region'] != 'World')].copy()
fig = px.area(df_region.sort_values(['region_es', 'year']), x='year', y='consumo_kwh', color='region_es', groupnorm='percent', labels={'year': 'Año', 'consumo_kwh': '%', 'region_es': 'Región'})
fig.add_vline(x=2026, line_dash='dash', line_color='black')
fig.add_annotation(x=2025.5, y=1.04, xref='x', yref='paper', text='Histórico', showarrow=False, xanchor='right')
fig.add_annotation(x=2026.5, y=1.04, xref='x', yref='paper', text='Proyección', showarrow=False, xanchor='left')
fig.update_layout(hovermode='x unified')
fig.show()

In [16]:
df = pd.read_csv('../data/merged_geothermal.csv')
df_melted = df[['Country', '2024 [1]', '2025 [2]', '2025 [3]']].copy()
df_melted.columns = ['País', '2024 [1]', '2025 [2]', '2025 [3]']
df_long = df_melted.melt(id_vars=['País'], var_name='Año', value_name='Capacidad [MW]')
fig = px.choropleth(df_long, locations='País', locationmode='country names', color='Capacidad [MW]', hover_name='País', animation_frame='Año', color_continuous_scale=px.colors.sequential.Plasma)
fig.layout.updatemenus[0].buttons[0].args[1]['frame']['duration'] = 1500
fig.layout.updatemenus[0].buttons[0].args[1]['transition']['duration'] = 500
fig.show()

/tmp/ipykernel_11634/788365236.py:7: DeprecationWarning:

The library used by the *country names* `locationmode` option is changing in an upcoming version. Country names in existing plots may not work in the new version. To ensure consistent behavior, consider setting `locationmode` to *ISO-3*.


In [17]:
df_geo = pd.read_csv('../data/merged_geothermal.csv')

top_10_2024 = df_geo[['Country', '2024 [1]']].sort_values(by='2024 [1]', ascending=False).head(10).reset_index(drop=True)
top_10_2025_2 = df_geo[['Country', '2025 [2]']].sort_values(by='2025 [2]', ascending=False).head(10).reset_index(drop=True)
top_10_2025_3 = df_geo[['Country', '2025 [3]']].sort_values(by='2025 [3]', ascending=False).head(10).reset_index(drop=True)

top_10 = pd.concat([top_10_2024, top_10_2025_2, top_10_2025_3], axis=1)
top_10.columns = ['País', '2024 [1]', 'País', '2025 [2]', 'País', '2025 [3]']
top_10